# AI Resume Intelligence — Model Rebuilding & Feature Engineering

This notebook organizes the raw CSV dataset, performs feature engineering and standardization, and trains an ML Job Role Classifier:
1. **Full-Null Cleansing**: Removes trailing/empty rows from `01_people.csv`, `03_education.csv`, `04_experience.csv`, `06_skills.csv`.
2. **Null Preservation**: Preserves candidate rows with null `email`, `phone`, `linkedin`.
3. **Column Pruning**: Drops `email`, `phone`, `linkedin`, `firm`, `location`, `institution`, `start_date` (from education).
4. **Date Standardization**: Converts `'Present'` -> `'08/2026'` and dates like `'August 2020'` -> `'08/2020'`.
5. **Experience Calculation**: Computes `end_date - start_date` in years.
6. **Feature Aggregation**: Merges skills, abilities, and education per candidate.
7. **Multi-Modal Feature Modeling**: Integrates both `profile_text` (TF-IDF) AND `total_experience_years` (StandardScaler) using `ColumnTransformer` for role prediction.
8. **ML Pipeline Training & Export**: Trains a `LinearSVC` model and exports `job_role_predictor.pkl`.

In [57]:
import os
import re
import pandas as pd
import numpy as np
from datetime import datetime
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Set dataset directory path
DATASET_DIR = './backend/dataset'
if not os.path.exists(DATASET_DIR):
    DATASET_DIR = './dataset'

print(f"Using dataset directory: {os.path.abspath(DATASET_DIR)}")

Using dataset directory: g:\Languages\Projects\AI Resume Intelligence\backend\dataset


### Step 1: Load Raw CSVs & Clean Trailing Null Rows

In [58]:
def load_and_clean_csv(filename):
    filepath = os.path.join(DATASET_DIR, filename)
    print(f"Loading {filename}...")
    df = pd.read_csv(filepath, low_memory=False)
    
    # Remove trailing rows that are completely NaN
    initial_len = len(df)
    df = df.dropna(how='all')
    
    # Ensure primary/foreign keys are valid integers
    if 'person_id' in df.columns:
        df = df[df['person_id'].notna()]
        df['person_id'] = df['person_id'].astype(int)
    elif 'skill' in df.columns:
        df = df[df['skill'].notna()]
        
    print(f"  -> {filename}: {initial_len:,} rows -> {len(df):,} cleaned rows.")
    return df

df_people = load_and_clean_csv("01_people_FINAL.csv")
df_abilities = load_and_clean_csv("02_abilities.csv")
df_education = load_and_clean_csv("03_education.csv")
df_experience = load_and_clean_csv("04_experience.csv")
df_person_skills = load_and_clean_csv("05_person_skills.csv")
df_skills_dict = load_and_clean_csv("06_skills.csv")

Loading 01_people_FINAL.csv...
  -> 01_people_FINAL.csv: 54,933 rows -> 54,933 cleaned rows.
Loading 02_abilities.csv...
  -> 02_abilities.csv: 1,219,473 rows -> 1,219,473 cleaned rows.
Loading 03_education.csv...
  -> 03_education.csv: 75,999 rows -> 75,999 cleaned rows.
Loading 04_experience.csv...
  -> 04_experience.csv: 265,404 rows -> 265,404 cleaned rows.
Loading 05_person_skills.csv...
  -> 05_person_skills.csv: 2,483,376 rows -> 2,483,376 cleaned rows.
Loading 06_skills.csv...
  -> 06_skills.csv: 226,760 rows -> 226,758 cleaned rows.


In [87]:
df_people["City"].value_counts()

City
Tiruchirappalli    1134
Ahmedabad          1114
Vadodara           1095
Chennai            1093
Belagavi           1093
                   ... 
Liverpool           363
Fort Worth          363
Gatineau            362
Sydney              354
Laval               334
Name: count, Length: 96, dtype: int64

In [ ]:
df_people["State"].value_counts()

State
Tamil Nadu          5360
Gujarat             5354
Karnataka           5284
Maharashtra         5160
Telangana           5157
Victoria            2035
Queensland          2023
Illinois            2010
Florida             2005
New York            1990
Texas               1980
Wales               1977
Scotland            1966
Ontario             1964
California          1952
New South Wales     1931
England             1927
British Columbia    1924
Quebec              1910
Delhi               1024
Name: count, dtype: int64

In [59]:
df_people[df_people['email'].notna()]

,person_id,Name,Role,email,phone,linkedin,City,State,Country
0,1,Aadhya Acharya,Database Administrator,aadhya.acharya@gmail.com,6416410688,NaN,New York City,New York,United States
1,2,Aadhya Adhikari,Database Administrator,aadhyaadhikari@gmail.com,8502258532,NaN,Townsville,Queensland,Australia
2,3,Aadhya Agarwal,Oracle Database Administrator,aagarwal32@gmail.com,6619610571,NaN,Gandhinagar,Gujarat,India
3,4,Aadhya Agnihotri,Amazon Redshift Administrator and ETL Develope...,aadhya.agnihotri@gmail.com,6122046974,NaN,Toronto,Ontario,Canada
4,5,Aadhya Ahuja,Scrum Master Scrum Master Scrum Master,aadhya.ahuja@gmail.com,9045873989,NaN,Rochester,New York,United States
...,...,...,...,...,...,...,...,...,...
54928,54929,Sameer Harris,Lead Python Developer,sameer.harris@gmail.com,9058907611,NaN,San Francisco,California,United States
54929,54930,Sameer Harrison,Full Stack Python Developer,sameer_harrison@gmail.com,7677616042,NaN,Mangaluru,Karnataka,India
54930,54931,Sameer Harvey,Eli Lilly,sharvey23@gmail.com,6210033798,NaN,Sunshine Coast,Queensland,Australia
54931,54932,Sameer Hayes,Python Developer,sameerhayes@gmail.com,6947370717,NaN,Tiruchirappalli,Tamil Nadu,India


In [60]:
df_people.isnull().sum()

person_id        0
Name             0
Role           114
email            0
phone            0
linkedin     46395
City             0
State            0
Country          0
dtype: int64

In [61]:
df_experience.head()

,person_id,title,firm,start_date,end_date,location
0,1,Database Administrator,Family Private Care LLC,04/2017,Present,"Roswell, GA"
1,1,Database Administrator,Incomm,01/2014,02/2017,"Alpharetta, GA"
2,2,Database Administrator,Intercontinental Registry,12/2008,08/2011,"Lagos, GU"
3,3,Oracle Database Administrator,Cognizant,06/2016,Present,"Hyderabad, Telangana"
4,3,Oracle Database Administrator,Convergys,06/2014,06/2016,"Hyderabad, Telangana"


In [62]:
firm_counts = df_experience["firm"].value_counts()

print(firm_counts.head(20))
print("Total rows:", len(df_experience))
print("Unique firms:", df_experience["firm"].nunique())

firm
AT&T                          1176
Wells Fargo                    915
IBM                            909
Bank of America                813
Verizon                        528
Comcast                        471
Capital One                    426
American Express               420
United States Marine Corps     417
Verizon Wireless               375
Accenture                      375
JP Morgan Chase                369
United States Air Force        336
Tata Consultancy Services      321
T-Mobile                       309
Self Employed                  303
Kaiser Permanente              303
Walmart                        291
United States Navy             288
Charter Communications         285
Name: count, dtype: int64
Total rows: 265404
Unique firms: 55352


In [63]:
df_education["program"].nunique

<bound method IndexOpsMixin.nunique of 0                                      Bachelor of Science
1                                  bsc in computer science
2        Master of Computer Applications in Science and...
3                             Bachelor in Computer Science
4                                                      NaN
                               ...                        
75994                   Master's in Information technology
75995                Bachelor's Degree in Computer Science
75996    Masters in Electrical and Electronics Engineering
75997                     Bachelor of Science in Computers
75998                                                  NaN
Name: program, Length: 75999, dtype: str>

### Step 2: Drop Unnecessary Metadata Columns for ML
* We keep all candidate rows (preserving candidates even if email/phone/linkedin are null)
* Drop columns: `email`, `phone`, `linkedin`, `firm`, `location`, `institution`, `start_date` (from education)

In [64]:
# !pip install reportlab

In [65]:
from reportlab.lib import pagesizes
# Drop metadata columns
df_people_clean = df_people.drop(columns=['email', 'phone', 'linkedin', 'Name'], errors='ignore')
df_education_clean = df_education.drop(columns=['institution', 'location', 'start_date'], errors='ignore')
df_experience_clean = df_experience.drop(columns=['firm', 'location'], errors='ignore')

print("People columns:", list(df_people_clean.columns))
print("Education columns:", list(df_education_clean.columns))
print("Experience columns:", list(df_experience_clean.columns))
print("Abilities columns:", list(df_abilities.columns))

People columns: ['person_id', 'Role', 'City', 'State', 'Country']
Education columns: ['person_id', 'program']
Experience columns: ['person_id', 'title', 'start_date', 'end_date']
Abilities columns: ['person_id', 'ability']


### Step 3: Date Standardization & Experience Duration Calculation
- Replace `'Present'`/`'Current'` in `end_date` with `'08/2026'`
- Standardize month names (e.g. `'August 2020'` -> `'08/2020'`)
- Calculate `exp_duration_years = (end_date - start_date)`

In [66]:
# 1. Directly replace 'Present' / 'Current' / 'Now' in the raw end_date column with '08/2026'
df_experience_clean['end_date'] = df_experience_clean['end_date'].astype(str).str.strip()
df_experience_clean['end_date'] = df_experience_clean['end_date'].replace(
    r'(?i)^\s*(present|current|now|ongoing|today)\s*$', '08/2026', regex=True
)
df_experience_clean['end_date'] = df_experience_clean['end_date'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# 2. Parsing Function for Date Objects
MONTH_MAP = {
    'jan': 1, 'january': 1, 'feb': 2, 'february': 2, 'mar': 3, 'march': 3,
    'apr': 4, 'april': 4, 'may': 5, 'jun': 6, 'june': 6, 'jul': 7, 'july': 7,
    'aug': 8, 'august': 8, 'sep': 9, 'september': 9, 'oct': 10, 'october': 10,
    'nov': 11, 'november': 11, 'dec': 12, 'december': 12
}

def parse_to_datetime(date_val, default_year=2026, default_month=8):
    if pd.isna(date_val) or date_val is None:
        return None
    val_str = str(date_val).strip()
    if not val_str or val_str.lower() in ['nan', 'none', 'null', '']:
        return None
        
    if re.search(r'present|current|now|ongoing', val_str, re.IGNORECASE):
        return datetime(default_year, default_month, 1)
        
    val_clean = re.sub(r'[,.-]', ' ', val_str).strip()
    parts = val_clean.split()
    
    # Month Name + Year e.g., 'August 2020'
    if len(parts) == 2:
        p0, p1 = parts[0].lower(), parts[1]
        if p0 in MONTH_MAP and p1.isdigit():
            y = int(p1)
            y = y + 2000 if y < 100 else y
            return datetime(y, MONTH_MAP[p0], 1)
        if p1.lower() in MONTH_MAP and p0.isdigit():
            y = int(p0)
            y = y + 2000 if y < 100 else y
            return datetime(y, MONTH_MAP[p1.lower()], 1)
        if p0.isdigit() and p1.isdigit():
            m, y = int(p0), int(p1)
            y = y + 2000 if y < 100 else y
            m = max(1, min(12, m))
            return datetime(y, m, 1)
            
    # 'MM/YYYY' or 'MM/YY'
    slash_parts = val_str.split('/')
    if len(slash_parts) == 2 and slash_parts[0].strip().isdigit() and slash_parts[1].strip().isdigit():
        m, y = int(slash_parts[0]), int(slash_parts[1])
        y = y + 2000 if y < 100 else y
        m = max(1, min(12, m))
        return datetime(y, m, 1)
        
    # Year only '2020'
    year_match = re.search(r'\b(19\d\d|20\d\d)\b', val_str)
    if year_match:
        return datetime(int(year_match.group(1)), 1, 1)
        
    return None

# Apply parsing
df_experience_clean['start_dt'] = df_experience_clean['start_date'].apply(parse_to_datetime)
df_experience_clean['end_dt'] = df_experience_clean['end_date'].apply(parse_to_datetime)

# Safe formatting that handles None and NaT properly without ValueError
df_experience_clean['start_date_std'] = df_experience_clean['start_dt'].apply(
    lambda d: d.strftime('%m/%Y') if (pd.notna(d) and d is not None) else np.nan
)
df_experience_clean['end_date_std'] = df_experience_clean['end_dt'].apply(
    lambda d: d.strftime('%m/%Y') if (pd.notna(d) and d is not None) else np.nan
)

# Calculate duration in years
def calc_duration(row):
    s, e = row['start_dt'], row['end_dt']
    if pd.isna(s) or pd.isna(e) or s is None or e is None:
        return 0.5
    try:
        diff_months = (e.year - s.year) * 12 + (e.month - s.month)
        return max(0.0, round(diff_months / 12.0, 2))
    except Exception:
        return 0.5

df_experience_clean['exp_duration_years'] = df_experience_clean.apply(calc_duration, axis=1)

print("Sample of standardized experience records:")
df_experience_clean[['person_id', 'title', 'start_date', 'start_date_std', 'end_date', 'end_date_std', 'exp_duration_years']].head(10)

Sample of standardized experience records:


,person_id,title,start_date,start_date_std,end_date,end_date_std,exp_duration_years
0,1,Database Administrator,04/2017,04/2017,08/2026,08/2026,9.33
1,1,Database Administrator,01/2014,01/2014,02/2017,02/2017,3.08
2,2,Database Administrator,12/2008,12/2008,08/2011,08/2011,2.67
3,3,Oracle Database Administrator,06/2016,06/2016,08/2026,08/2026,10.17
4,3,Oracle Database Administrator,06/2014,06/2014,06/2016,06/2016,2.00
5,4,Amazon Redshift Administrator and ETL Develope...,02/18,02/2018,08/2026,08/2026,8.50
6,4,Database Administrator,11/14,11/2014,12/15,12/2015,1.08
7,4,Database Administrator,09/07,09/2007,10/14,10/2014,7.08
8,5,Scrum Master,10/2015,10/2015,04/2019,04/2019,3.50
9,5,Oracle Database Administrator/ Scrum Master,06/2013,06/2013,10/2015,10/2015,2.33


### Step 4: Aggregate Features per Candidate

In [67]:
# 1. Total Experience per candidate (Summed up across all jobs)
df_total_exp = df_experience_clean.groupby('person_id')['exp_duration_years'].sum().reset_index()
df_total_exp.rename(columns={'exp_duration_years': 'total_experience_years'}, inplace=True)

# 2. Latest Job Title per candidate
df_exp_sorted = df_experience_clean.sort_values(by=['person_id', 'end_dt'], ascending=[True, False])
df_latest_title = df_exp_sorted.drop_duplicates(subset=['person_id'], keep='first')[['person_id', 'title']]
df_latest_title.rename(columns={'title': 'latest_job_title'}, inplace=True)

# 3. Aggregate Skills per person
df_agg_skills = df_person_skills.groupby('person_id')['skill'].apply(lambda x: ' '.join(str(s) for s in x if pd.notna(s))).reset_index()
df_agg_skills.rename(columns={'skill': 'aggregated_skills'}, inplace=True)

# 4. Aggregate Abilities per person
ability_col = 'ability' if 'ability' in df_abilities.columns else 'description'
df_agg_abilities = df_abilities.groupby('person_id')[ability_col].apply(lambda x: ' '.join(str(d) for d in x if pd.notna(d))).reset_index()
df_agg_abilities.rename(columns={ability_col: 'aggregated_abilities'}, inplace=True)
# 5. Aggregate Education Programs per person
import re
def get_degree_level(program_str):
    text_lower = str(program_str).lower()
    master_patterns = [r'\bmaster\b', r'\bmsc\b', r'\bm\.sc\b', r'\bmtech\b', r'\bm\.tech\b', r'\bms\b', r'\bm\.s\b', r'\bma\b', r'\bm\.a\b', r'\bmba\b']
    if any(re.search(pat, text_lower) for pat in master_patterns): return 2.0
    bachelor_patterns = [r'\bbachelor\b', r'\bbsc\b', r'\bb\.sc\b', r'\bbtech\b', r'\bb\.tech\b', r'\bbs\b', r'\bb\.s\b', r'\bba\b', r'\bb\.a\b', r'\bbba\b']
    if any(re.search(pat, text_lower) for pat in bachelor_patterns): return 1.0
    return 0.0

df_education_clean['degree_level'] = df_education_clean['program'].apply(get_degree_level)
df_max_degree = df_education_clean.groupby('person_id')['degree_level'].max().reset_index()

df_agg_edu = df_education_clean.groupby('person_id')['program'].apply(lambda x: ' '.join(str(p) for p in x if pd.notna(p))).reset_index()
df_agg_edu.rename(columns={'program': 'aggregated_education'}, inplace=True)
df_agg_edu = df_agg_edu.merge(df_max_degree, on='person_id', how='left')


print("Feature aggregation completed successfully!")

Feature aggregation completed successfully!


In [68]:
df_latest_title.head(3)

,person_id,latest_job_title
0,1,Database Administrator
2,2,Database Administrator
3,3,Oracle Database Administrator


### Step 5: Merge Master Candidate Dataset

In [69]:
# Left join with people dataset to retain all 54K candidates
df_master = df_people_clean.merge(df_latest_title, on='person_id', how='left')
df_master = df_master.merge(df_total_exp, on='person_id', how='left')
df_master = df_master.merge(df_agg_skills, on='person_id', how='left')
df_master = df_master.merge(df_agg_abilities, on='person_id', how='left')
df_master = df_master.merge(df_agg_edu, on='person_id', how='left')

# Fill missing numerical and text values
df_master['total_experience_years'] = df_master['total_experience_years'].fillna(0.0)
df_master['aggregated_skills'] = df_master['aggregated_skills'].fillna('')
df_master['aggregated_abilities'] = df_master['aggregated_abilities'].fillna('')
df_master['aggregated_education'] = df_master['aggregated_education'].fillna('')

# Target Role label
df_master['target_role'] = df_master['latest_job_title'].fillna(df_master['Role'])

# Combined Profile Text for TF-IDF
df_master['profile_text'] = (
    df_master['aggregated_skills'] + ' ' + 
    df_master['aggregated_abilities'] + ' ' + 
    df_master['aggregated_education']
).str.strip()

print(f"Master Dataset Shape: {df_master.shape}")
print("Experience stats across all candidates:")
print(df_master['total_experience_years'].describe())
df_master[['person_id', 'target_role', 'total_experience_years', 'profile_text']].head(5)

Master Dataset Shape: (54933, 13)
Experience stats across all candidates:
count    54933.000000
mean        16.504688
std          9.968028
min          0.000000
25%         11.250000
50%         14.910000
75%         19.850000
max        144.480000
Name: total_experience_years, dtype: float64


,person_id,target_role,total_experience_years,profile_text
0,1,Database Administrator,12.41,Database administration Database Ms sql server...
1,2,Database Administrator,2.67,sql server management studio visual studio sql...
2,3,Oracle Database Administrator,12.17,DATABASES ORACLE (4 years) ORACLE 10G SQL LINU...
3,4,Amazon Redshift Administrator and ETL Develope...,16.66,Maintain multiple database environments (Redsh...
4,5,Scrum Master,6.91,Scrum Agile software development Product backl...


In [70]:
df_master["degree_level"].map({
    1.0: "Bachelor",
    2.0: "Master",
    0.0: "Other/None"
}).value_counts()


degree_level
Bachelor      20313
Other/None    17511
Master        10251
Name: count, dtype: int64

In [71]:
df_master.head(2)

,person_id,Role,City,State,Country,latest_job_title,total_experience_years,aggregated_skills,aggregated_abilities,aggregated_education,degree_level,target_role,profile_text
0,1,Database Administrator,New York City,New York,United States,Database Administrator,12.41,Database administration Database Ms sql server...,Installation and Building Server Running Backu...,Bachelor of Science,1.0,Database Administrator,Database administration Database Ms sql server...
1,2,Database Administrator,Townsville,Queensland,Australia,Database Administrator,2.67,sql server management studio visual studio sql...,database management systems administration dev...,bsc in computer science,1.0,Database Administrator,sql server management studio visual studio sql...


In [72]:
# 1. Total number of unique values
df_education_clean["degree_level"].nunique()

3

### Step 6: Train Multi-Modal ML Model (Text + Experience Years)
We use a `ColumnTransformer` to combine:
1. **`profile_text`**: Processed with `TfidfVectorizer` (captures skills, tools, and domain abilities).
2. **`total_experience_years`**: Scaled with `StandardScaler` (enables the model to learn seniority and experience-level role distinctions, e.g., Senior vs Junior roles).
3. **Classifier**: `LinearSVC` for multi-class role classification.

In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [74]:
# Filter dataset with valid profile text and target role
df_model_data = df_master[
    (df_master['profile_text'].str.len() > 15) & 
    (df_master['target_role'].notna()) & 
    (df_master['target_role'].str.strip() != '')
].copy()

df_model_data['target_role'] = df_model_data['target_role'].str.title().str.strip()
top_roles = df_model_data['target_role'].value_counts().head(50).index
df_train_subset = df_model_data[df_model_data['target_role'].isin(top_roles)].copy()

print(f"Training on {len(df_train_subset):,} samples across top {len(top_roles)} job roles.")

# Multi-Modal Feature Matrix
X = df_train_subset[['profile_text', 'total_experience_years', 'degree_level']]
y = df_train_subset['target_role']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Multi-Feature Preprocessor: TF-IDF for Text + StandardScaler for Experience Years
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(ngram_range=(1, 2),max_features=10000,stop_words='english',sublinear_tf=True), 'profile_text'),
        ('exp', StandardScaler(), ['total_experience_years'])
    ]
)

# Build Multi-Modal Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LinearSVC(C=1.0,random_state=42,max_iter=3000))
])

print("Training Pipeline with Text and Experience Years features...")
pipeline.fit(X_train, y_train)

# Evaluation
y_pred = pipeline.predict(X_test)
print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report Summary:")
print(classification_report(y_test, y_pred, zero_division=0))

Training on 18,771 samples across top 50 job roles.
Training Pipeline with Text and Experience Years features...

Model Accuracy: 97.52%

Classification Report Summary:
                                   precision    recall  f1-score   support

                Android Developer       1.00      1.00      1.00        20
                       Consultant       1.00      1.00      1.00        22
           Cyber Security Analyst       1.00      1.00      1.00        52
           Database Administrator       1.00      0.97      0.98       258
              Front End Developer       0.93      1.00      0.96       259
          Front End Web Developer       0.90      1.00      0.95        56
             Front- End Developer       0.95      0.96      0.95        54
              Front-End Developer       0.94      0.85      0.89        59
          Front-End Web Developer       1.00      1.00      1.00        21
             Full Stack Developer       1.00      1.00      1.00        49
     

In [75]:
df_model_data.head(2)

,person_id,Role,City,State,Country,latest_job_title,total_experience_years,aggregated_skills,aggregated_abilities,aggregated_education,degree_level,target_role,profile_text
0,1,Database Administrator,New York City,New York,United States,Database Administrator,12.41,Database administration Database Ms sql server...,Installation and Building Server Running Backu...,Bachelor of Science,1.0,Database Administrator,Database administration Database Ms sql server...
1,2,Database Administrator,Townsville,Queensland,Australia,Database Administrator,2.67,sql server management studio visual studio sql...,database management systems administration dev...,bsc in computer science,1.0,Database Administrator,sql server management studio visual studio sql...


### Step 7: Export Model to Disk & Test Prediction

In [76]:
OUTPUT_DIR = './backend/ml_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'job_role_predictor.pkl')

joblib.dump(pipeline, output_path)
print(f"Saved trained model to: {output_path}")

# Validation Test with DataFrame input containing both profile text and total_experience_years
sample_candidate = pd.DataFrame({
    'profile_text': ['Python, PyTorch, SQL, Pandas, ETL pipelines, NLP, Machine Learning, Deep Learning'],
    'total_experience_years': [6.5]
})

predicted_role = pipeline.predict(sample_candidate)[0]
print(f"\nSample Candidate Input:")
print(f"  - Text: {sample_candidate['profile_text'].iloc[0]}")
print(f"  - Experience: {sample_candidate['total_experience_years'].iloc[0]} years")
print(f"👉 Predicted Job Role: {predicted_role}")

Saved trained model to: ./backend/ml_models\job_role_predictor.pkl

Sample Candidate Input:
  - Text: Python, PyTorch, SQL, Pandas, ETL pipelines, NLP, Machine Learning, Deep Learning
  - Experience: 6.5 years
👉 Predicted Job Role: Software Engineer
